# 🏛️ MARINERA POWER: Pipeline Industrial Definitivo & Control Humano (Lotes de 20)

Pipeline oficial de profesionalización de catálogo para **Marinera Power**.
- **Fondo de Estudio:** Gris cálido `#E5E1DE` con sombra de contacto y luz difusa 5200K.
- **Pre-procesamiento:** Corrección EXIF + Detección de bordes Sobel (umbral 3.5) + Redimensionamiento Lanczos a 1920px.
- **Control Humano (Human-in-the-Loop):** Procesamiento en **lotes de 20 fotos** para revisión y feedback de Carlos Carbajal.
- **Respaldo Doble:** Enrutamiento en espejo a carpetas PRO / Erróneas + Bóveda Central permanente.
- **Tablero Web en Vivo:** https://carloscarbajallinan.github.io/marinera-studio-pro/


In [ ]:
# 1. Montar Google Drive & Dependencias Industriales
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U pillow pillow-heif requests google-auth gspread opencv-python-headless numpy
import pillow_heif
pillow_heif.register_heif_opener()
print('✅ Google Drive montado y dependencias industriales listas.')

In [ ]:
# 2. Autenticación, Conexión Sheets y Auto-Detección de Rutas en Drive
import os, io, gc, json, base64, time, requests, shutil
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageOps
from google.colab import userdata, auth
from IPython.display import display, Image as IPImage
import gspread
from google.auth import default

# 1. Cargar API Key
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ GEMINI_API_KEY cargada con éxito desde Secrets.')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Ingresa tu Gemini API Key: ')

# 2. Conexión Google Sheets
auth.authenticate_user()
creds, _ = default()
gc_client = gspread.authorize(creds)
SPREADSHEET_ID = '1kRsCZ7ZpPxyxrSEkUrYRl3NHahrQQche90RKK2yedQ8'
sh = gc_client.open_by_key(SPREADSHEET_ID)
sheet_control = sh.worksheet('Control_Total_Drive_V2')
print(f'✅ Conexión establecida con la hoja maestra: {sheet_control.title}')

# 3. Auto-detección de la ruta exacta en Google Drive
candidates_base = [
    Path('/content/drive/MyDrive/Emprendimiento/Marinera power 2026'),
    Path('/content/drive/MyDrive/Marinera power 2026'),
    Path('/content/drive/MyDrive'),
]

BASE_DRIVE = Path('/content/drive/MyDrive')
for c in candidates_base:
    if (c / 'Fotos catalogo').exists():
        BASE_DRIVE = c
        break

DIR_ORIGEN_ROOT = BASE_DRIVE / 'Fotos catalogo'
DIR_PRO_ROOT = BASE_DRIVE / 'Fotos catalogo PRO'
DIR_ERROR_ROOT = DIR_PRO_ROOT / '_Fotos_Erroneas_y_Desalineadas'
BOVEDA_BACKUP_DIR = DIR_PRO_ROOT / '_BOVEDA_HISTORIAL_BACKUP'
CHECKPOINT_FILE = DIR_PRO_ROOT / 'checkpoint_pipeline.json'

DIR_PRO_ROOT.mkdir(parents=True, exist_ok=True)
DIR_ERROR_ROOT.mkdir(parents=True, exist_ok=True)
BOVEDA_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f'📁 Ruta Origen detectada:  {DIR_ORIGEN_ROOT}')
print(f'📁 Ruta PRO detectada:     {DIR_PRO_ROOT}')
print(f'📁 Ruta Erróneas en Drive: {DIR_ERROR_ROOT}')
print(f'📁 Bóveda de Respaldo:     {BOVEDA_BACKUP_DIR}')

# 4. Resolvedor Robusto Linux (partes_limpias + búsqueda insensible)
def resolve_photo_path(ruta_origen, orig_name):
    """
    Localiza la foto física en Google Drive de forma ultra-robusta.
    Limpia espacios en rutas tipo Linux y realiza búsqueda insensible a extensiones.
    """
    partes = [p.strip() for p in str(ruta_origen).split('/') if p.strip()]
    if partes and partes[0].lower() in ['fotos catalogo', 'fotos catalogo marinera power']:
        partes = partes[1:]
        
    foto_dir = DIR_ORIGEN_ROOT.joinpath(*partes)
    
    clean_name = str(orig_name).replace('.jpg', '').replace('.jpeg', '').replace('.png', '').replace('.heic', '').replace('.HEIC', '').strip()
    
    # 1. Búsqueda directa en la subcarpeta del producto
    if foto_dir.exists():
        cands = list(foto_dir.glob(f"{clean_name}*"))
        if cands:
            return cands[0]
            
    # 2. Fallback global insensible en todo el árbol de origen
    global_cands = list(DIR_ORIGEN_ROOT.glob(f"**/{clean_name}*"))
    if global_cands:
        return global_cands[0]
        
    return None

print('✅ Resolvedor de rutas Linux y búsqueda insensible compilado.')



In [ ]:
# 3. Pre-Procesamiento Sobel (Umbral 3.5) & Context Managers Anti-OOM
def detect_and_crop_artificial_bars_sobel(img_rgb: Image.Image, max_scan_pct: float = 0.25):
    arr = np.array(img_rgb)
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.hypot(grad_x, grad_y)
    
    max_h = int(h * max_scan_pct)
    max_w = int(w * max_scan_pct)
    
    def is_artificial_band(slice_gray: np.ndarray, slice_mag: np.ndarray) -> bool:
        return float(np.std(slice_gray)) < 3.5 and float(np.mean(slice_mag)) < 3.5
        
    top = 0
    for r in range(max_h):
        if is_artificial_band(gray[r, :], mag[r, :]): top = r + 1
        else: break
        
    bottom = h
    for r in range(h - 1, h - max_h, -1):
        if is_artificial_band(gray[r, :], mag[r, :]): bottom = r
        else: break
        
    left = 0
    for c in range(max_w):
        if is_artificial_band(gray[:, c], mag[:, c]): left = c + 1
        else: break
        
    right = w
    for c in range(w - 1, w - max_w, -1):
        if is_artificial_band(gray[:, c], mag[:, c]): right = c
        else: break
        
    top = top if top >= 12 else 0
    bottom = bottom if (h - bottom) >= 12 else h
    left = left if left >= 12 else 0
    right = right if (w - right) >= 12 else w
    
    if top > 0 or bottom < h or left > 0 or right > w:
        return img_rgb.crop((left, top, right, bottom)), True
    return img_rgb, False

def prepare_image_pipeline(file_path: Path, max_dim: int = 1920):
    with Image.open(file_path) as raw_img:
        transposed = ImageOps.exif_transpose(raw_img)
        base = transposed if transposed is not None else raw_img
        rgb = base.convert('RGB')
        clean_img, was_cropped = detect_and_crop_artificial_bars_sobel(rgb)
        
        w, h = clean_img.size
        if max(w, h) > max_dim:
            scale = max_dim / max(w, h)
            final_img = clean_img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
        else:
            final_img = clean_img.copy()
            
        del rgb, clean_img
        gc.collect()
        return final_img, was_cropped

print('✅ Pre-Procesamiento Sobel (Umbral 3.5) y Lanczos 1920 compilados.')

In [ ]:
# 4. Motor de Generación & Dual Quality Gate (Determinístico en OpenCV + Visión IA)
import numpy as np
import cv2

PROMPT_MAESTRO = """[TASK DIRECTIVE]
Professionally retouch this commercial fashion and jewelry photograph for the high-end e-commerce catalog of Marinera Power. Modify ONLY the background and floor surfaces. Do NOT alter the subject.

[IMMUTABLE COLOR & STUDIO UNIFORMITY - HEX #E5E1DE]
- BACKDROP COLOR: The entire background must be an absolutely uniform, solid, flat warm-grey studio backdrop matching hex color #E5E1DE (exact RGB: 229, 225, 222).
- ZERO GRADIENTS & ZERO VIGNETTING: Completely eliminate room corners, wall seams, domestic shadows, and color gradients. The background tone must be 100% consistent across all catalog photos. Zero beige shifts, zero brown casts, zero cool white clinical shifts.
- ZERO COLOR BOUNCE & AMBIENT SPILL: Highly saturated red, purple, fuchsia garments and gold jewelry must NOT bleed color, radiosity, or ambient spill onto the background.

[MANDATORY SCALE & BOUNDING BOX LOCK - NO ZOOM OUT / NO SHRINKING]
- PRESERVE 1:1 SCALE: The subject (model, dancer, garment, jewelry) MUST maintain the EXACT same pixel dimensions, proportion, and framing as in the original photograph.
- STRICTLY PROHIBITED TO ZOOM OUT: Do NOT push the subject into the distance. Do NOT shrink the subject. The subject must remain in full foreground focus exactly as framed originally.
- NO OUTPAINTING: If the subject is cut at the waist, bust, or legs, KEEP that exact crop. Do NOT hallucinate missing limbs or expand the field of view.

[CLEAN FLOOR & FOOT CONTACT - ZERO DIRT OR SMUDGES]
- PRISTINE FLOOR: The ground surface must be clean and flawless.
- NATURAL CONTACT SHADOW: Render ONLY a delicate, soft, feathered, transparent contact shadow directly under the soles of shoes or bare feet. 
- ABSOLUTELY NO DIRT, NO MUDDY PATCHES, NO DARK SMUDGES, and NO grey halos around the feet or hemline.

[PRESERVATION OF DETAILS]
- Faces, skin tone, hair, white linen marinera handkerchief, and 18k Peruvian gold filigree must remain 100% untouched and razor-sharp."""

def edit_background_gemini(pil_img, prompt_text=PROMPT_MAESTRO, max_retries=3):
    buffered = io.BytesIO()
    pil_img.save(buffered, format='JPEG', quality=95)
    b64_image = base64.b64encode(buffered.getvalue()).decode('utf-8')

    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-image:generateContent?key={API_KEY}'
    payload = {
        'contents': [{
            'parts': [
                {'text': prompt_text},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_image}}
            ]
        }],
        'generationConfig': {
            'responseModalities': ['IMAGE'],
            'temperature': 0.05
        }
    }

    for intento in range(max_retries):
        try:
            res = requests.post(url, json=payload, timeout=90)
            if res.status_code == 200:
                parts = res.json()['candidates'][0]['content']['parts']
                for p in parts:
                    if 'inlineData' in p:
                        img_bytes = base64.b64decode(p['inlineData']['data'])
                        return Image.open(io.BytesIO(img_bytes)), None
            elif res.status_code in (429, 503):
                time.sleep(12 * (intento + 1))
                continue
            else:
                return None, f'HTTP {res.status_code}: {res.text[:120]}'
        except requests.exceptions.Timeout:
            if intento == max_retries - 1:
                return None, 'Timeout API (90s alcanzado)'
            time.sleep(5)
        except Exception as e:
            return None, str(e)
    return None, 'Fallo recurrente tras reintentos'

def deterministic_quality_gates(orig_pil, pro_pil):
    """
    Quality Gates Deterministas en Python (< 50ms, Costo $0):
    1. QG-01: Delta Cromático Periférico en 4 esquinas vs #E5E1DE (RGB 229, 225, 222)
    2. QG-02: Bounding Box & Escala 1:1 (Detección de Zoom-out / encogimiento)
    3. QG-03: Desplazamiento de Centroide (Desbalance de composición)
    4. QG-05: Índice de Limpieza de Piso (Detección de manchas oscuras / barro)
    """
    np_orig = np.array(orig_pil)
    np_pro = np.array(pro_pil)
    h, w = np_pro.shape[:2]

    # 1. Delta Cromático en 4 esquinas
    target_rgb = np.array([229, 225, 222], dtype=np.float32)
    corner_sz = max(15, min(w, h) // 15)
    corners = [
        np_pro[0:corner_sz, 0:corner_sz],
        np_pro[0:corner_sz, -corner_sz:],
        np_pro[h//2 - corner_sz:h//2, 0:corner_sz],
        np_pro[h//2 - corner_sz:h//2, -corner_sz:]
    ]
    mean_corner_rgb = np.mean([c.mean(axis=(0,1)) for c in corners], axis=0)
    delta_color = float(np.linalg.norm(mean_corner_rgb - target_rgb))

    # 2. Escala 1:1 y Bounding Box
    dist_orig = np.linalg.norm(np_orig.astype(np.float32) - np_orig[0:15, 0:15].mean(axis=(0,1)), axis=2)
    dist_pro = np.linalg.norm(np_pro.astype(np.float32) - target_rgb, axis=2)
    mask_orig = (dist_orig > 25).astype(np.uint8)
    mask_pro = (dist_pro > 25).astype(np.uint8)

    scale_ratio = 1.0
    centroid_shift = 0.0
    pts_orig = cv2.findNonZero(mask_orig)
    pts_pro = cv2.findNonZero(mask_pro)

    if pts_orig is not None and pts_pro is not None:
        _, _, _, h_orig_box = cv2.boundingRect(pts_orig)
        _, _, _, h_pro_box = cv2.boundingRect(pts_pro)
        scale_ratio = float(h_pro_box) / float(h_orig_box) if h_orig_box > 0 else 1.0
        m_pro = cv2.moments(mask_pro)
        if m_pro['m00'] > 0:
            cx_pro = m_pro['m10'] / m_pro['m00']
            centroid_shift = abs(cx_pro - (w / 2.0)) / float(w)

    # 3. Limpieza de Piso (15% inferior)
    floor_roi = np_pro[int(h * 0.85):, :]
    floor_dist = np.linalg.norm(floor_roi.astype(np.float32) - target_rgb, axis=2)
    dirt_pixels = np.sum((floor_dist > 45) & (np.mean(floor_roi, axis=2) < 160)) / float(floor_roi.shape[0] * floor_roi.shape[1])
    floor_dirt_score = max(0, 100 - int(dirt_pixels * 500))

    # Veredicto Determinístico
    det_pass = True
    fail_reasons = []

    if scale_ratio < 0.85:
        det_pass = False
        fail_reasons.append(f"Zoom-out no autorizado: Sujeto encogido al {int(scale_ratio*100)}%")
    elif scale_ratio > 1.15:
        fail_reasons.append(f"Encuadre expandido / Outpainting: {scale_ratio:.2f}x")

    if delta_color > 18.0:
        det_pass = False
        fail_reasons.append(f"Desvío cromático: ΔE={delta_color:.1f} (Límite 18.0)")

    if floor_dirt_score < 60:
        fail_reasons.append("Piso sucio / Manchas oscuras bajo los pies")

    score_color = max(0, 100 - int(delta_color * 3.5))
    score_scale = max(10, int(scale_ratio * 100)) if scale_ratio < 0.85 else 100
    score_det = int(score_color * 0.35 + score_scale * 0.40 + floor_dirt_score * 0.15 + (100 - int(centroid_shift * 100)) * 0.10)
    if not det_pass:
        score_det = min(score_det, 35)

    return {
        'score_deterministico': score_det,
        'aprobada_deterministico': det_pass,
        'delta_color': round(delta_color, 1),
        'escala_ratio': round(scale_ratio, 2),
        'desplazamiento_centroide': round(centroid_shift, 3),
        'limpieza_piso_score': floor_dirt_score,
        'motivo': " | ".join(fail_reasons) if fail_reasons else "Cumple todos los Quality Gates matemáticos (#E5E1DE, escala 1:1, piso limpio)"
    }

def audit_image_quality(orig_pil, pro_pil):
    """
    Auditoría Híbrida:
    1. Quality Gate Matemático (60% peso + Veto Absoluto)
    2. Opinión Visión Multimodal Gemini 2.5 Flash (40% peso)
    """
    det_eval = deterministic_quality_gates(orig_pil, pro_pil)

    # Veto Inmediato sin complacencia si falla el determinismo:
    if not det_eval['aprobada_deterministico']:
        return {
            'score': det_eval['score_deterministico'],
            'score_det': det_eval['score_deterministico'],
            'score_ia': 0,
            'aprobada': False,
            'motivo': f"🚨 VETO DETERMINÍSTICO: {det_eval['motivo']}",
            'detalles_det': det_eval
        }

    # Si pasa el determinismo, solicitamos la opinión de Gemini Vision:
    buf1, buf2 = io.BytesIO(), io.BytesIO()
    orig_pil.save(buf1, format='JPEG', quality=85)
    pro_pil.save(buf2, format='JPEG', quality=85)
    b64_1 = base64.b64encode(buf1.getvalue()).decode('utf-8')
    b64_2 = base64.b64encode(buf2.getvalue()).decode('utf-8')

    eval_prompt = (
        'Actúa como Auditor Riguroso de E-Commerce para Marinera Power. Evalúa la foto PRO respecto a la Original:\n'
        '1. ¿Manos, dedos, pañuelo de marinera y filigrana de oro están 100% nítidos e intactos?\n'
        '2. ¿La sombra de contacto es natural, suave y transparente?\n'
        'Responde estrictamente en JSON: {"score": 0-100, "aprobada": true/false, "motivo": "breve resumen"}'
    )
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}'
    payload = {
        'contents': [{
            'parts': [
                {'text': eval_prompt},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_1}},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_2}}
            ]
        }],
        'generationConfig': {'responseMimeType': 'application/json'}
    }

    score_ia = 80
    aprobada_ia = True
    motivo_ia = "Evaluación IA satisfactoria"

    try:
        res = requests.post(url, json=payload, timeout=25)
        if res.status_code == 200:
            ai_data = json.loads(res.json()['candidates'][0]['content']['parts'][0]['text'])
            score_ia = ai_data.get('score', 80)
            aprobada_ia = ai_data.get('aprobada', True)
            motivo_ia = ai_data.get('motivo', 'Validado')
    except Exception as e:
        motivo_ia = f"Fallo parcial de visión: {e}"

    # Ponderación Oficial: 60% Determinístico + 40% IA
    score_final = int(det_eval['score_deterministico'] * 0.60 + score_ia * 0.40)
    aprobada_final = det_eval['aprobada_deterministico'] and aprobada_ia and (score_final >= 80)

    return {
        'score': score_final,
        'score_det': det_eval['score_deterministico'],
        'score_ia': score_ia,
        'aprobada': aprobada_final,
        'motivo': f"[Det: {det_eval['score_deterministico']}/100 | IA: {score_ia}/100] {motivo_ia}",
        'detalles_det': det_eval
    }

print('✅ Motor de Generación & Dual Quality Gate (Determinístico + IA) cargados.')



In [ ]:
# 5. Módulo de Enrutamiento en Espejo & Checkpointing Persistente
def save_and_route_photo(pro_img, row_data, qa_result):
    row_id = str(row_data.get('ID', '0'))
    orig_name = str(row_data.get('Nombre_Archivo_Original', 'foto'))
    clean_name = orig_name.replace('.jpg', '').replace('.jpeg', '').replace('.png', '').replace('.heic', '').replace('.HEIC', '')
    score = qa_result.get('score', 85)
    aprobada = qa_result.get('aprobada', True) and (score >= 80)

    # Bóveda Central Permanente
    bov_path = BOVEDA_BACKUP_DIR / f'ID{row_id.zfill(3)}_{clean_name}_score{score}.jpg'
    pro_img.save(bov_path, quality=95)

    # Enrutamiento en Espejo
    ruta_origen = str(row_data.get('Ruta_Carpeta_Drive', row_data.get('Carpeta_Ruta_Original', '')))
    subrama = ruta_origen.replace('Fotos catalogo / ', '').replace('Fotos catalogo/', '').strip()

    if aprobada:
        target_dir = DIR_PRO_ROOT / subrama
        final_path = target_dir / f'{clean_name}_pro.jpg'
    else:
        target_dir = DIR_ERROR_ROOT / f'{subrama}_Error'
        final_path = target_dir / f'{clean_name}_QA_FALLIDA.jpg'

    target_dir.mkdir(parents=True, exist_ok=True)
    pro_img.save(final_path, quality=95)
    return str(final_path), str(bov_path), aprobada

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                return json.load(f)
        except Exception:
            pass
    return {'processed_ids': []}

def save_checkpoint(processed_ids):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'processed_ids': processed_ids, 'last_update': time.strftime('%Y-%m-%d %H:%M:%S')}, f, indent=2)

print('✅ Enrutamiento en Espejo y Checkpointing persistente listos.')

In [ ]:
# 6. CANARY TEST DETERMINISTA (5 Casos Críticos de Fuego)
canary_targets = [
    {
        'id_test': 'CANARY_1_BARRAS_BLANCAS',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Lila con cinta morada',
        'orig_name': 'foto (3).png',
        'caso': 'Recorte Sobel de barras laterales blancas de Instagram (193px)'
    },
    {
        'id_test': 'CANARY_2_PARQUET_SALON',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Lila con cinta morada',
        'orig_name': '34a63e97 PRO.jpg',
        'caso': 'Reemplazo total de suelo de parquet por ciclorama #E5E1DE con oclusión'
    },
    {
        'id_test': 'CANARY_3_PAÑUELO_MANO',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Coral con cinta blanca',
        'orig_name': 'd72049af PRO.jpg',
        'caso': 'Blindaje de mano y pañuelo de marinera al viento (cero amputación)'
    },
    {
        'id_test': 'CANARY_4_FILIGRANA_JOYA',
        'subpath': 'Para ella/Accesorios/Aretes para Marinera',
        'orig_name': '1.jpg',
        'caso': 'Macro joyería: Oro peruano 18k y filigrana calada con brillo especular'
    },
    {
        'id_test': 'CANARY_5_CORTE_ENQUADRE',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Morada con cinta blanca',
        'orig_name': '1841c6d0 PRO.jpg',
        'caso': 'Mandato Cliente: Cero outpainting en plano recortado (conservar encuadre)'
    }
]

print(f"🦅 INICIANDO CANARY TEST DETERMINISTA (Ruta Base: {DIR_ORIGEN_ROOT})...\n")
canary_results = []

for idx, tgt in enumerate(canary_targets, start=1):
    test_file = resolve_photo_path(tgt['subpath'], tgt['orig_name'])
    
    if not test_file or not test_file.exists():
        print(f"⚠️ [{idx}/5] No se localizó archivo para {tgt['id_test']}: {tgt['orig_name']}")
        continue

    print(f"▶️ [{idx}/5] Evaluando {tgt['id_test']}: {test_file.name}...")
    print(f"   🎯 Caso a probar: {tgt['caso']}")
    
    t0 = time.time()
    try:
        prep_img, was_cropped = prepare_image_pipeline(test_file)
        pro_img, err = edit_background_gemini(prep_img)
        
        if pro_img:
            qa = audit_image_quality(prep_img, pro_img)
            final_p, bov_p, aprobada = save_and_route_photo(pro_img, {
                'ID': f'CANARY_{idx}',
                'Nombre_Archivo_Original': test_file.name,
                'Ruta_Carpeta_Drive': tgt['subpath']
            }, qa)
            t_tot = time.time() - t0
            
            print(f"   ✅ Score QA: {qa.get('score')}/100 | Aprobada: {aprobada} | Total: {t_tot:.1f}s (Sobel Crop: {was_cropped})")
            print(f"   📝 Motivo QA: {qa.get('motivo')}")
            print(f"   📁 Guardada en: {final_p}")
            canary_results.append({'test': tgt['id_test'], 'score': qa.get('score'), 'aprobada': aprobada, 'file': test_file.name})
            display(pro_img)
        else:
            print(f"   ❌ Fallo API: {err}")
    except Exception as exc:
        print(f"   ❌ Error procesando {test_file.name}: {exc}")
    time.sleep(3)

aprobadas = sum(1 for c in canary_results if c.get('aprobada'))
print(f"\n🏆 RESUMEN CANARY TEST: {aprobadas}/{len(canary_results)} Aprobadas.")
if aprobadas >= 4:
    print("🚀 ¡CANARY TEST SUPERADO CON ÉXITO! Las fotos cumplen todos los Quality Gates del Red Team.")



In [ ]:
# ==============================================================================
# 🎯 PASO 7: PROCESAMIENTO EN LOTES DE 20 FOTOS (HUMAN-IN-THE-LOOP)
# ==============================================================================
# 🔒 SEGURO DE ACTIVACIÓN MANUAL:
# Para ejecutar el Lote 1 (20 fotos), cambia esta variable a True:
EJECUTAR_LOTE_PRODUCCION = False

CHUNK_SIZE = 20  # Lote estricto de 20 fotos solicitado por Carlos Carbajal

if not EJECUTAR_LOTE_PRODUCCION:
    print("🔒 [PASO 7 EN ESPERA] Seguro de activación manual activo.")
    print("👉 Este paso NO se ejecutará al pulsar 'Ejecutar todo'.")
    print(f"👉 Para procesar el siguiente lote de {CHUNK_SIZE} fotos:")
    print("   1. Cambia arriba: EJECUTAR_LOTE_PRODUCCION = True")
    print("   2. Presiona únicamente el botón Play de esta celda.")
else:
    print(f"🚀 INICIANDO LOTE DE {CHUNK_SIZE} FOTOS (Mesa de Control Oficial)...")
    
    # 1. Cargar datos de la pestaña oficial: Control_Total_Drive_V2
    sheet_data = sheet_control.get_all_records()
    headers = sheet_control.row_values(1)
    
    # Column indices (1-based for gspread)
    col_estado_idx = headers.index('Estado') + 1 if 'Estado' in headers else 8
    
    # Asegurar columnas para feedback de Carlos
    if 'Feedback_Carlos' not in headers:
        sheet_control.update_cell(1, len(headers) + 1, 'Feedback_Carlos')
        sheet_control.update_cell(1, len(headers) + 2, 'Detalle_Error_Carlos')
        headers = sheet_control.row_values(1)
        
    col_feedback_idx = headers.index('Feedback_Carlos') + 1
    col_detalle_idx = headers.index('Detalle_Error_Carlos') + 1
    
    # 2. Cargar Checkpoint
    checkpoint = load_checkpoint()
    processed_set = set(checkpoint.get('processed_ids', []))
    print(f"📦 Fotos ya aseguradas en checkpoint previo: {len(processed_set)}")
    
    # 3. Filtrar pendientes
    pendientes = []
    for idx, row in enumerate(sheet_data, start=2):
        row_id = str(row.get('ID', ''))
        estado = str(row.get('Estado', '')).upper()
        if row_id and (row_id not in processed_set) and ('PRO_VERIFICADO' not in estado):
            pendientes.append((idx, row))
            
    print(f"📊 Total fotos pendientes en catálogo: {len(pendientes)}")
    chunk_to_process = pendientes[:CHUNK_SIZE]
    print(f"🎯 Procesando lote de {len(chunk_to_process)} fotos ahora...\n")
    
    exitosas_chunk = 0
    rechazadas_chunk = 0
    
    for i, (row_idx, row_data) in enumerate(chunk_to_process, start=1):
        row_id = str(row_data.get('ID', '0'))
        orig_name = str(row_data.get('Nombre_Archivo_Original', ''))
        ruta_origen = str(row_data.get('Ruta_Carpeta_Drive', row_data.get('Carpeta_Ruta_Original', '')))
        
        print(f"[{i}/{len(chunk_to_process)}] Foto ID {row_id}: {orig_name}")
        
        # Localización ultra-robusta
        foto_path = resolve_photo_path(ruta_origen, orig_name)
        if not foto_path or not foto_path.exists():
            print(f"   ⚠️ Archivo físico no hallado en Drive. Saltando.")
            continue
            
        t0 = time.time()
        try:
            # Pre-procesamiento Sobel
            prep_img, was_cropped = prepare_image_pipeline(foto_path)
            
            # Generación IA
            pro_img, err = edit_background_gemini(prep_img)
            
            if pro_img:
                # Auditoría de Calidad
                qa = audit_image_quality(prep_img, pro_img)
                final_path, bov_path, aprobada = save_and_route_photo(pro_img, row_data, qa)
                
                # Registro en Sheet
                nuevo_estado = 'PRO_PENDIENTE_FEEDBACK_CARLOS' if aprobada else 'FALLIDA_QA'
                sheet_control.update_cell(row_idx, col_estado_idx, nuevo_estado)
                
                if aprobada:
                    exitosas_chunk += 1
                    print(f"   ✅ Aprobada QA (Score {qa.get('score')}/100) -> Guardada en PRO y Bóveda")
                else:
                    rechazadas_chunk += 1
                    print(f"   ⚠️ Rechazada QA (Score {qa.get('score')}/100) -> Guardada en Carpeta Erróneas")
                    
                display(pro_img)
            else:
                sheet_control.update_cell(row_idx, col_estado_idx, 'ERROR_GENERACION_API')
                print(f"   ❌ Fallo API Gemini: {err}")
                
        except Exception as exc:
            print(f"   ❌ Excepción aislada en ID {row_id}: {exc}")
        finally:
            gc.collect()
            
        processed_set.add(row_id)
        save_checkpoint(list(processed_set))
        time.sleep(3)
        
    print(f"\n==============================================================================")
    print(f"🎉 LOTE DE {len(chunk_to_process)} FOTOS COMPLETADO!")
    print(f"📊 Resumen: {exitosas_chunk} aprobadas por QA | {rechazadas_chunk} en carpeta erróneas")
    print(f"📁 Todas las fotos tienen copia de seguridad en: _BOVEDA_HISTORIAL_BACKUP")
    print(f"🌐 Tablero de Control Web: https://carloscarbajallinan.github.io/marinera-studio-pro/")
    print(f"📋 Hoja Oficial en Sheets: {sheet_control.title}")
    print(f"👉 Asigna tu feedback (APROBADO_CARLOS / CORREGIR_CARLOS) en el Tablero o Sheet")
    print(f"   antes de correr el siguiente lote.")
    print(f"==============================================================================")

